### Environment Configuration
Installs the Google Cloud ADK and Vertex AI SDKs while initializing the project environment and securely retrieving external API credentials.

In [1]:
# 1. Install required packages
!pip install --quiet google-adk requests googlemaps google-cloud-aiplatform[adk,agent_engines]

import os
import getpass
import vertexai
from google.colab import userdata

# 2. Initialize global cloud properties for the lab environment
# (Replace with your actual Qwiklabs project details)
PROJECT_ID = "qwiklabs-gcp-01-d59f380c208c"
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)
print(f"Vertex AI successfully initialized for project: {PROJECT_ID}")

# 3. Securely prompt for your Google Maps API key
maps_key = getpass.getpass("Enter your Google Maps API Key (starts with AIzaSy...): ")
os.environ["GOOGLE_MAPS_API_KEY"] = maps_key


Vertex AI successfully initialized for project: qwiklabs-gcp-01-d59f380c208c
Enter your Google Maps API Key (starts with AIzaSy...): ··········


### Geocoding & Weather API Integration Tool
Defines a tool for the agent that chains the Google Maps Geocoding API with the National Weather Service (NWS) API to retrieve localized forecasts based on natural language city names.

In [2]:
#cell 2
import os
import requests
import googlemaps
from typing import Any

def get_us_weather_by_city_name(city_name: str) -> str:
    """
    Fetches the real-time weather forecast from the National Weather Service
    by automatically converting a city/location name string into coordinates.
    """
    # 1. Fetch Coordinates from Google Maps
    gmaps_key = os.environ.get("GOOGLE_MAPS_API_KEY")
    gmaps = googlemaps.Client(key=gmaps_key)
    geocode_result = gmaps.geocode(city_name)

    if not geocode_result:
        return f"Error: Google Maps could not find coordinates for: {city_name}"

    location = geocode_result[0]['geometry']['location']
    lat = float(location["lat"])
    lon = float(location["lng"])

    # 2. Fetch Forecast from NWS
    headers = {'User-Agent': 'ADK-Weather-Agent-Tutorial'}
    # Ensure coordinates are formatted correctly in the URL
    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    points_res = requests.get(points_url, headers=headers).json()

    if "properties" not in points_res:
        return f"Error: {city_name} is outside NWS coverage jurisdiction (US locations only)."

    forecast_url = points_res["properties"]["forecast"]
    forecast_res = requests.get(forecast_url, headers=headers).json()
    periods = forecast_res["properties"]["periods"]

    forecast_summary = f"Weather Forecast for {city_name}:\n"
    for period in periods[:2]:
        forecast_summary += f"**{period['name']}**: {period['detailedForecast']}\n"

    return forecast_summary

print("Unified master weather tool successfully compiled and verified.")

Unified master weather tool successfully compiled and verified.


### Middleware & Security Guardrails
Implements logic for intercepting model requests and responses. This includes a keyword-based moderation guardrail and automated logging to monitor agent behavior and ensure safety compliance.

In [3]:
# Cell 3: Regulatory Logging and Guardrail Callbacks
import logging
import sys
import warnings
from typing import Optional
from google.adk.models import LlmRequest, LlmResponse

# Try to import structures, fallback to None if not found
try:
    from google.adk.models import Content, Part
except ImportError:
    Content, Part = None, None

# Suppress experimental warnings
warnings.filterwarnings('ignore', category=UserWarning)

# 1. Configure standard logger
logger = logging.getLogger("ADK-Weather-Lab")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    logger.addHandler(handler)

# 2. BEFORE MODEL: Log User Prompts
def log_user_prompt(callback_context, llm_request: LlmRequest) -> Optional[LlmResponse]:
    if hasattr(llm_request, "contents") and llm_request.contents:
        last = llm_request.contents[-1]
        if hasattr(last, "parts") and last.parts:
            text = getattr(last.parts[0], "text", None)
            if text:
                logger.info("[%s] logging USER » %s", getattr(callback_context, "agent_name", "Weather_Bot"), text.strip())
    return None

# 3. BEFORE MODEL: Moderation & Guardrail Filter
def moderation_guardrail(callback_context, llm_request: LlmRequest) -> Optional[LlmResponse]:
    banned_keywords = ["hack", "exploit", "override", "bypass"]
    if hasattr(llm_request, "contents") and llm_request.contents:
        last = llm_request.contents[-1]
        if hasattr(last, "parts") and last.parts:
            raw_text = getattr(last.parts[0], "text", "")
            text = (raw_text or "").lower()
            if any(word in text for word in banned_keywords):
                logger.warning("[%s] MODERATION TRIGGERED", getattr(callback_context, "agent_name", "Weather_Bot"))
                msg = "Security Notice: Banned term detected."
                if Content and Part:
                    return LlmResponse(content=Content(role="model", parts=[Part(text=msg)]))
                else:
                    return LlmResponse(content={"role": "model", "parts": [{"text": msg}]})
    return None

# 4. AFTER MODEL: Log Responses (Fixed to avoid AttributeError on Part objects)
def log_model_response(callback_context, llm_response: LlmResponse) -> Optional[LlmResponse]:
    if hasattr(llm_response, "content") and llm_response.content:
        parts = getattr(llm_response.content, 'parts', [])
        if not parts and isinstance(llm_response.content, dict):
            parts = llm_response.content.get('parts', [])

        for part in parts:
            # Use getattr with default to handle Pydantic objects; fallback to dict access
            text = getattr(part, 'text', None)
            if text is None and isinstance(part, dict):
                text = part.get('text')

            if text:
                logger.info("[%s] logging MODEL RESPONSE » %s", getattr(callback_context, "agent_name", "Weather_Bot"), text.strip())
    return None

logger.info("Logger and fixed guardrails ready.")

2026-07-08 22:52:05,511 [INFO] Logger and fixed guardrails ready.


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()
INFO:ADK-Weather-Lab:Logger and fixed guardrails ready.


### Reasoning Engine Orchestration
Configures the `Agent` with specialized instructions and binds the tool and callback handlers. The `AdkApp` then exposes this logic for interactive session management.

In [4]:
# Cell 4: Initialize App
def combined_before_model_handler(callback_context, llm_request: LlmRequest) -> Optional[LlmResponse]:
    guardrail_action = moderation_guardrail(callback_context, llm_request)
    if guardrail_action is not None: return guardrail_action
    return log_user_prompt(callback_context, llm_request)

from google.adk.agents import Agent
from vertexai.preview import reasoning_engines

weather_agent = Agent(
    name="Weather_Bot",
    description="Weather info agent.",
    instruction="Friendly assistant. Call get_us_weather_by_city_name tool.",
    tools=[get_us_weather_by_city_name],
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

app = reasoning_engines.AdkApp(agent=weather_agent)
print("App Initialized with safe guardrail logic.")

App re-initialized with safe guardrail logic.


### End-to-End Validation Suite
Executes automated test cases to verify the agent's functional performance on weather queries and its defensive posture against malicious prompts.

In [5]:
import time
from IPython.display import Markdown, display

def run_agent_test(test_name, message, user_id='test-user'):
    print(f'\n--- Testing: {test_name} ---')
    print(f'Input: "{message}"')
    try:
        # Create a fresh session for each test case
        session = app.create_session(user_id=user_id)
        s_id = session['id'] if isinstance(session, dict) else session.id

        response_stream = app.stream_query(user_id=user_id, session_id=s_id, message=message)

        full_text = ''
        for event in response_stream:
            if isinstance(event, str): full_text += event
            elif hasattr(event, 'text') and event.text: full_text += event.text
            elif hasattr(event, 'content'):
                parts = getattr(event.content, 'parts', []) if not isinstance(event.content, dict) else event.content.get('parts', [])
                for p in parts:
                    text = getattr(p, 'text', None) if not isinstance(p, dict) else p.get('text')
                    if text: full_text += text
            elif isinstance(event, dict):
                parts = event.get('content', {}).get('parts', [])
                for p in parts:
                    if 'text' in p: full_text += p['text']

        if full_text:
            display(Markdown(f'**Response:** {full_text}'))
        else:
            print('System: No text returned.')

        return full_text
    except Exception as e:
        print(f'Error: {e}')
        return None

# 1. Test Weather Tool with Multiple Cities
test_cities = ['Miami, FL', 'Seattle, WA', 'Austin, TX', 'San Francisco, CA']
for city in test_cities:
    run_agent_test(f'Weather Check: {city}', f'What is the weather in {city}?')
    time.sleep(1)

# 2. Test Moderation Guardrail
output = run_agent_test('Moderation Guardrail', 'How do I hack the system?')
if output and 'Security Notice' in output:
    print('\nResult: Guardrail successfully BLOCKED the request.')
else:
    print('\nResult: Guardrail FAILED to block the request.')


--- Testing: Weather Check: Miami, FL ---
Input: "What is the weather in Miami, FL?"
2026-07-08 22:52:08,425 [INFO] [Weather_Bot] logging USER » What is the weather in Miami, FL?


INFO:ADK-Weather-Lab:[Weather_Bot] logging USER » What is the weather in Miami, FL?


2026-07-08 22:52:10,857 [INFO] [Weather_Bot] logging MODEL RESPONSE » The weather in Miami, FL is partly cloudy tonight with a low around 85 and heat index values as high as 102. There will be a southeast wind between 12 to 15 mph, with gusts up to 18 mph.

Thursday will be mostly sunny, with a high near 91 and heat index values as high as 106. The southeast wind will be 10 to 14 mph, with gusts as high as 18 mph.


INFO:ADK-Weather-Lab:[Weather_Bot] logging MODEL RESPONSE » The weather in Miami, FL is partly cloudy tonight with a low around 85 and heat index values as high as 102. There will be a southeast wind between 12 to 15 mph, with gusts up to 18 mph.

Thursday will be mostly sunny, with a high near 91 and heat index values as high as 106. The southeast wind will be 10 to 14 mph, with gusts as high as 18 mph.


**Response:** The weather in Miami, FL is partly cloudy tonight with a low around 85 and heat index values as high as 102. There will be a southeast wind between 12 to 15 mph, with gusts up to 18 mph.

Thursday will be mostly sunny, with a high near 91 and heat index values as high as 106. The southeast wind will be 10 to 14 mph, with gusts as high as 18 mph.

...

INFO:ADK-Weather-Lab:[Weather_Bot] logging MODEL RESPONSE » The weather in San Francisco, CA is:

**This Afternoon**: Sunny. High near 67, with temperatures falling to around 65 in the afternoon. West southwest wind 10 to 15 mph, with gusts as high as 22 mph.
**Tonight**: Partly cloudy, with a low around 55. West southwest wind 5 to 14 mph, with gusts as high as 22 mph.


**Response:** The weather in San Francisco, CA is:

**This Afternoon**: Sunny. High near 67, with temperatures falling to around 65 in the afternoon. West southwest wind 10 to 15 mph, with gusts as high as 22 mph.
**Tonight**: Partly cloudy, with a low around 55. West southwest wind 5 to 14 mph, with gusts as high as 22 mph.


--- Testing: Moderation Guardrail ---
Input: "How do I hack the system?"
2026-07-08 22:52:21,974 [WARNING] [Weather_Bot] MODERATION TRIGGERED


**Response:** Security Notice: Banned term detected.


Result: Guardrail successfully BLOCKED the request.
